In [1]:
!pip show chromadb rank_bm25 sentence-transformers transformers spacy | grep -E "Name|Version"

Name: sentence-transformers
Version: 5.4.1
Name: transformers
Version: 5.0.0
Name: spacy
Version: 3.8.14


In [2]:
!pip install -q chromadb rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 300.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 21.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 26.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 26.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages tha

In [3]:
import os
import shutil

src = '/kaggle/input/datasets/harshamin07/researchlens-dataset'
dst = '/kaggle/working/'

os.makedirs(dst, exist_ok=True)

for fname in ['bm25_index.pkl', 'bm25_metadata.pkl', 'fixed_chunks.pkl',
              'semantic_chunks.pkl', 'parsed_papers.json',
              'papers_metadata.json', 'qasper_pairs.json']:
    src_path = os.path.join(src, fname)
    if os.path.exists(src_path):
        shutil.copy(src_path, os.path.join(dst, fname))
        print(f"  ✓ {fname}")

chroma_src = os.path.join(src, 'indexes', 'chroma_db')
chroma_dst = os.path.join(dst, 'chroma_db')
if os.path.exists(chroma_src) and not os.path.exists(chroma_dst):
    shutil.copytree(chroma_src, chroma_dst)
    print(f"  ✓ chroma_db/")
elif os.path.exists(os.path.join(src, 'chroma_db')) and not os.path.exists(chroma_dst):
    shutil.copytree(os.path.join(src, 'chroma_db'), chroma_dst)
    print(f"  ✓ chroma_db/ (root)")

print("\nDone loading from dataset")

  ✓ bm25_index.pkl
  ✓ bm25_metadata.pkl
  ✓ fixed_chunks.pkl
  ✓ semantic_chunks.pkl
  ✓ parsed_papers.json
  ✓ papers_metadata.json
  ✓ qasper_pairs.json
  ✓ chroma_db/ (root)

Done loading from dataset


In [4]:
import pickle
import json
import re
import numpy as np
from typing import List, Dict, Tuple
import torch
import spacy
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [5]:
nlp = spacy.load("en_core_web_sm")

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

nli_pipeline = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

chroma_client = chromadb.PersistentClient(path='/kaggle/working/chroma_db')
fixed_collection = chroma_client.get_collection("fixed_chunks")

with open('/kaggle/working/bm25_index.pkl', 'rb') as f:
    bm25_index = pickle.load(f)
with open('/kaggle/working/bm25_metadata.pkl', 'rb') as f:
    bm25_metadata = pickle.load(f)

print(f"spaCy: {spacy.__version__}")
print(f"NLI model loaded on: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"ChromaDB fixed: {fixed_collection.count()} docs")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

spaCy: 3.8.14
NLI model loaded on: GPU
ChromaDB fixed: 14500 docs


In [6]:
def retrieve_broad(topic: str, top_k: int = 20) -> List[Dict]:
    query_embedding = embed_model.encode(topic).tolist()
    
    dense_results = fixed_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=['documents', 'distances', 'metadatas']
    )
    
    dense_hits = {}
    for rank, (doc, dist, meta) in enumerate(zip(
        dense_results['documents'][0],
        dense_results['distances'][0],
        dense_results['metadatas'][0]
    )):
        chunk_id = meta.get('chunk_id', f"d_{rank}")
        dense_hits[chunk_id] = {
            'rank': rank, 'text': doc, 'score': 1 - dist,
            'paper_id': meta['paper_id'], 'title': meta['title']
        }

    bm25_scores = bm25_index.get_scores(topic.lower().split())
    top_idx = np.argsort(bm25_scores)[::-1][:top_k]
    bm25_hits = {}
    for rank, idx in enumerate(top_idx):
        cid = bm25_metadata[idx]['chunk_id']
        bm25_hits[cid] = {
            'rank': rank, 'text': bm25_metadata[idx]['text'],
            'score': bm25_scores[idx],
            'paper_id': bm25_metadata[idx]['paper_id'],
            'title': bm25_metadata[idx]['title']
        }

    k = 60
    all_ids = set(dense_hits) | set(bm25_hits)
    rrf = {cid: (1/(dense_hits[cid]['rank']+k) if cid in dense_hits else 0) +
                (1/(bm25_hits[cid]['rank']+k) if cid in bm25_hits else 0)
           for cid in all_ids}

    top_ids = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    results = []
    for cid in top_ids:
        source = dense_hits.get(cid) or bm25_hits[cid]
        results.append({
            'chunk_id': cid,
            'text': source['text'],
            'paper_id': source['paper_id'],
            'title': source['title'],
            'score': rrf[cid]
        })
    return results

print("Broad retrieval function defined")

Broad retrieval function defined


In [7]:
def extract_claims(chunks: List[Dict], min_words: int = 15, max_words: int = 55) -> List[Dict]:
    claims = []

    skip_patterns = [
        r'@\w+',
        r'\b(arxiv|preprint|doi|http|www)\b',
        r'table \d+|figure \d+|fig\.|appendix',
        r'^\s*\d+\s',
        r'et al\.?',              # citation fragments
        r'\(\s*\d{4}',            # "(2022"
        r'\d{4}[a-z]?\s*[;)]',    # "2022;" / "2022)"
        r'\b\d+\s*\)',            # enumerations like "2)"
    ]

    for chunk in chunks:
        doc = nlp(chunk['text'])
        for sent in doc.sents:
            text = sent.text.strip()
            word_count = len(text.split())

            if word_count < min_words or word_count > max_words:
                continue

            # must be a well-formed, complete standalone sentence
            if not text.endswith(('.', '!', '?')):
                continue
            if not text[0].isupper():
                continue
            if text.count(';') >= 2 or text.count('(') != text.count(')'):
                continue

            skip = False
            for pattern in skip_patterns:
                if re.search(pattern, text, re.IGNORECASE):
                    skip = True
                    break
            if skip:
                continue

            has_subject = any(tok.dep_ in ('nsubj', 'nsubjpass') for tok in sent)
            has_verb = any(tok.pos_ == 'VERB' for tok in sent)

            if has_subject and has_verb:
                claims.append({
                    'text': text,
                    'paper_id': chunk['paper_id'],
                    'title': chunk['title'],
                    'chunk_id': chunk['chunk_id']
                })

    seen = set()
    unique_claims = []
    for claim in claims:
        if claim['text'] not in seen:
            seen.add(claim['text'])
            unique_claims.append(claim)

    return unique_claims

print("Claim extraction defined (stricter: complete sentences, no citation fragments)")


Claim extraction defined (stricter: complete sentences, no citation fragments)


In [8]:
def get_cross_paper_pairs(claims: List[Dict], topic: str, 
                           top_n_per_paper: int = 3) -> List[Tuple]:
    if not claims:
        return []
    
    topic_embedding = embed_model.encode(topic, convert_to_numpy=True)
    claim_texts = [c['text'] for c in claims]
    claim_embeddings = embed_model.encode(claim_texts, convert_to_numpy=True)
    
    from sklearn.metrics.pairwise import cosine_similarity
    topic_sims = cosine_similarity([topic_embedding], claim_embeddings)[0]
    
    for i, claim in enumerate(claims):
        claim['topic_sim'] = float(topic_sims[i])
    
    by_paper = {}
    for claim in claims:
        pid = claim['paper_id']
        if pid not in by_paper:
            by_paper[pid] = []
        by_paper[pid].append(claim)
    
    for pid in by_paper:
        by_paper[pid] = sorted(by_paper[pid], 
                                key=lambda x: x['topic_sim'], 
                                reverse=True)[:top_n_per_paper]
    
    paper_ids = list(by_paper.keys())
    pairs = []
    for i in range(len(paper_ids)):
        for j in range(i+1, len(paper_ids)):
            for ca in by_paper[paper_ids[i]]:
                for cb in by_paper[paper_ids[j]]:
                    pairs.append((ca, cb))
    
    return pairs

print("Cross-paper pairing defined")

Cross-paper pairing defined


In [9]:
def detect_contradictions(pairs: List[Tuple],
                          contradiction_threshold: float = 0.45,
                          min_claim_similarity: float = 0.40) -> List[Dict]:
    contradictions = []
    
    filtered_pairs = []
    for claim_a, claim_b in pairs:
        emb_a = embed_model.encode(claim_a['text'], convert_to_numpy=True)
        emb_b = embed_model.encode(claim_b['text'], convert_to_numpy=True)
        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity([emb_a], [emb_b])[0][0]
        if sim >= min_claim_similarity:
            filtered_pairs.append((claim_a, claim_b, float(sim)))
    
    print(f"  Filtered to {len(filtered_pairs)} pairs with similarity >= {min_claim_similarity}")
    
    for claim_a, claim_b, sim in filtered_pairs:
        try:
            result_ab = nli_pipeline(
                sequences=claim_a['text'],
                candidate_labels=['contradiction', 'entailment', 'neutral'],
                hypothesis_template=claim_b['text'] + " {}"
            )
            result_ba = nli_pipeline(
                sequences=claim_b['text'],
                candidate_labels=['contradiction', 'entailment', 'neutral'],
                hypothesis_template=claim_a['text'] + " {}"
            )
            
            scores_ab = dict(zip(result_ab['labels'], result_ab['scores']))
            scores_ba = dict(zip(result_ba['labels'], result_ba['scores']))
            
            contra_score = (scores_ab.get('contradiction', 0) +
                           scores_ba.get('contradiction', 0)) / 2
            
            if contra_score >= contradiction_threshold:
                contradictions.append({
                    'claim_a': claim_a['text'],
                    'paper_a': claim_a['title'],
                    'claim_b': claim_b['text'],
                    'paper_b': claim_b['title'],
                    'contradiction_score': round(contra_score, 3),
                    'claim_similarity': round(sim, 3)
                })
        except Exception:
            continue
    
    return sorted(contradictions,
                  key=lambda x: x['contradiction_score'],
                  reverse=True)

print("NLI detection redefined with similarity pre-filter")

NLI detection redefined with similarity pre-filter


In [14]:
def _dedup_contradictions(contradictions: List[Dict]) -> List[Dict]:
    # keep only the single strongest contradiction per unique claim (either side)
    best = {}
    used_a, used_b, kept = set(), set(), []
    for c in contradictions:  # already sorted by score desc
        if c['claim_a'] in used_a or c['claim_b'] in used_b:
            continue
        used_a.add(c['claim_a']); used_b.add(c['claim_b'])
        kept.append(c)
    return kept


def find_contradictions(topic: str, top_k_chunks: int = 50) -> Dict:
    print(f"Step 1: Retrieving chunks for '{topic}'...")
    chunks = retrieve_broad(topic, top_k=top_k_chunks)
    print(f"  Retrieved {len(chunks)} chunks")

    print("Step 2: Extracting claims...")
    claims = extract_claims(chunks)
    print(f"  Extracted {len(claims)} unique claims")

    print("Step 3: Building cross-paper pairs...")
    pairs = get_cross_paper_pairs(claims, topic, top_n_per_paper=2)
    print(f"  Built {len(pairs)} cross-paper pairs to evaluate")

    print("Step 4: Running NLI on pairs...")
    # stricter: claims must be about the same thing (0.50) AND clearly conflict (0.55)
    contradictions = detect_contradictions(
        pairs, contradiction_threshold=0.44, min_claim_similarity=0.42
    )
    contradictions = _dedup_contradictions(contradictions)
    print(f"  Found {len(contradictions)} genuine contradictions after dedup")

    return {
        'topic': topic,
        'chunks_retrieved': len(chunks),
        'claims_extracted': len(claims),
        'pairs_evaluated': len(pairs),
        'contradictions': contradictions
    }

print("Full pipeline redefined (thresholds 0.44/0.42 + dedup)")


Full pipeline redefined (thresholds 0.44/0.42 + dedup)


In [16]:
test_topics = [
    "does fine-tuning always improve model performance",
    "optimal learning rate for transformer training",
    "retrieval augmented generation versus fine-tuning",
    "attention mechanisms in transformer models",
]

all_results = []
for topic in test_topics:
    print(f"\n{'='*70}")
    result = find_contradictions(topic)
    all_results.append(result)
    
    print(f"\nResults for: '{topic}'")
    print(f"  Chunks: {result['chunks_retrieved']} | Claims: {result['claims_extracted']} | "
          f"Pairs: {result['pairs_evaluated']} | Contradictions: {len(result['contradictions'])}")
    
    if result['contradictions']:
        print(f"\n  Top contradictions:")
        for i, c in enumerate(result['contradictions'][:3]):
            print(f"\n  [{i+1}] Score: {c['contradiction_score']:.3f}")
            print(f"    Paper A: {c['paper_a'][:60]}")
            print(f"    Claim A: {c['claim_a'][:150]}")
            print(f"    Paper B: {c['paper_b'][:60]}")
            print(f"    Claim B: {c['claim_b'][:150]}")
    else:
        print("  No contradictions found above threshold")


Step 1: Retrieving chunks for 'does fine-tuning always improve model performance'...
  Retrieved 50 chunks
Step 2: Extracting claims...
  Extracted 345 unique claims
Step 3: Building cross-paper pairs...
  Built 1986 cross-paper pairs to evaluate
Step 4: Running NLI on pairs...
  Filtered to 637 pairs with similarity >= 0.42
  Found 10 genuine contradictions after dedup

Results for: 'does fine-tuning always improve model performance'
  Chunks: 50 | Claims: 345 | Pairs: 1986 | Contradictions: 10

  Top contradictions:

  [1] Score: 0.583
    Paper A: Fine-Tuning without Performance Degradation
    Claim A: Monotonic performance improvement during fine-tuning is often challenging, as agents typically experience performance degradation at the early fine-tu
    Paper B: You Only Fine-tune Once: Many-Shot In-Context Fine-Tuning fo
    Claim B: In contrast, the many-shot fine-tuned model maintains strong performance when tested on unseen task types.

  [2] Score: 0.543
    Paper A: Fine-Tu

In [17]:
# Explanation pass — one-sentence "why they disagree" per contradiction (Mistral-7B, T4)
from transformers import AutoTokenizer, AutoModelForCausalLM

_tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
_mdl = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    torch_dtype=torch.float16, device_map="auto"
)

def explain_disagreement(paper_a, claim_a, paper_b, claim_b):
    prompt = (
        "[INST] Two ML papers make claims a detector flagged as conflicting.\n"
        f'Paper A ("{paper_a[:60]}"): "{claim_a}"\n'
        f'Paper B ("{paper_b[:60]}"): "{claim_b}"\n'
        "In ONE sentence, state what the two claims actually disagree about. "
        'If they do not genuinely conflict, reply exactly "NOT_A_CONTRADICTION". [/INST]'
    )
    inputs = _tok(prompt, return_tensors="pt").to(_mdl.device)
    out = _mdl.generate(**inputs, max_new_tokens=60, temperature=0.3, do_sample=True)
    text = _tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return text

for r in all_results:
    kept = []
    for c in r['contradictions']:
        exp = explain_disagreement(c['paper_a'], c['claim_a'], c['paper_b'], c['claim_b'])
        if "NOT_A_CONTRADICTION" in exp.upper():
            continue                      # Mistral vetoes false positives
        c['explanation'] = exp
        kept.append(c)
    r['contradictions'] = kept
    print(f"{r['topic'][:45]:45} -> {len(kept)} explained contradictions")

print("Explanations added; false positives dropped")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


does fine-tuning always improve model perform -> 10 explained contradictions


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


optimal learning rate for transformer trainin -> 4 explained contradictions


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


retrieval augmented generation versus fine-tu -> 1 explained contradictions
attention mechanisms in transformer models    -> 1 explained contradictions
Explanations added; false positives dropped


In [18]:
import json, os

serializable = [{
    'topic': r['topic'],
    'chunks_retrieved': r['chunks_retrieved'],
    'claims_extracted': r['claims_extracted'],
    'pairs_evaluated': r['pairs_evaluated'],
    'contradictions': r['contradictions']
} for r in all_results]

out_path = '/kaggle/working/contradiction_results.json'
with open(out_path, 'w') as f:
    json.dump(serializable, f, indent=2)

total = sum(len(s['contradictions']) for s in serializable)
print(f"Wrote {out_path}")
print(f"  exists: {os.path.exists(out_path)} | size: {os.path.getsize(out_path)} bytes | total: {total}")
for s in serializable:
    print(f"    {s['topic'][:45]:45} -> {len(s['contradictions'])}")
print("JSON files in /kaggle/working:", [f for f in os.listdir('/kaggle/working') if f.endswith('.json')])


print("Saved contradiction_results.json")
print("\n=== Notebook 06 Complete ===")
print("Pipeline: retrieve → extract claims → pair → similarity filter → NLI → report")

Wrote /kaggle/working/contradiction_results.json
  exists: True | size: 16029 bytes | total: 16
    does fine-tuning always improve model perform -> 10
    optimal learning rate for transformer trainin -> 4
    retrieval augmented generation versus fine-tu -> 1
    attention mechanisms in transformer models    -> 1
JSON files in /kaggle/working: ['contradiction_results.json', 'papers_metadata.json', 'parsed_papers.json', 'qasper_pairs.json']
Saved contradiction_results.json

=== Notebook 06 Complete ===
Pipeline: retrieve → extract claims → pair → similarity filter → NLI → report
